# QSAR XGBoost parameter testing

This notebook tests selected XGBoost hyperparameters for the QSAR classification task. It provides a controlled comparison of parameter settings.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

MODEL_DIR = "xgboost"
MODEL_NAME = "XGBoost"

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qsar_config import DATA_PATH, RANDOM_SEED
from qsar_common import (
    add_mol_column,
    aggregate_targets_by_fingerprint,
    build_morgan_fingerprints,
    encode_targets,
    load_qsar_dataset,
    stack_features_and_targets,
    to_target_probability_matrix,
)

In [2]:
df = load_qsar_dataset(DATA_PATH)

In [3]:
df = add_mol_column(df, smiles_column="Smiles", mol_column="mol")

In [4]:
FPSIZE = 1024
df = build_morgan_fingerprints(
    df,
    mol_column="mol",
    output_column="fp",
    radius=2,
    n_bits=FPSIZE,
)

In [5]:
df, encoder, target_names = encode_targets(
    df,
    target_column="Target",
    output_column="target_encoded",
)

In [6]:
df_agg = aggregate_targets_by_fingerprint(
    df,
    fp_column="fp",
    encoded_target_column="target_encoded",
    aggregated_target_column="target",
)

In [7]:
x, y = stack_features_and_targets(
    df_agg,
    fp_column="fp",
    target_column="target",
)

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=RANDOM_SEED,
)

In [ ]:
n_estimators_list = [1, 2, 5, 10, 20, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000]

n_targets = y_test.shape[1]
history = {i: [] for i in range(n_targets)}

print(f"Starting XGBoost training loop on {len(n_estimators_list)} configurations...")

for n in n_estimators_list:
    xgb_estimator = XGBClassifier(
        n_estimators=n,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        n_jobs=-1,
        random_state=RANDOM_SEED,
        tree_method="hist",
        eval_metric="logloss",
    )

    model = OneVsRestClassifier(xgb_estimator)
    model.fit(x_train, y_train)

    y_pred_prob = model.predict_proba(x_test)
    y_pred_prob_matrix = to_target_probability_matrix(
        y_pred_prob,
        n_targets=n_targets,
    )

    for i in range(n_targets):
        try:
            score = roc_auc_score(y_test[:, i], y_pred_prob_matrix[:, i])
        except ValueError:
            score = 0.5
        history[i].append(score)

    print(f"n_estimators={n} done.", flush=True)

print("\nLoop finished.")

Starting XGBoost training loop on 21 configurations...
n_estimators=1 done.
n_estimators=2 done.
n_estimators=5 done.
n_estimators=10 done.
n_estimators=20 done.
n_estimators=50 done.
n_estimators=70 done.
n_estimators=100 done.
n_estimators=150 done.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

plt.figure(figsize=(12, 8))
cmap = plt.colormaps.get_cmap('tab10')

czech_names = {
    "ar": "androgenní receptor",
    "pr": "progesteronový receptor",
    "gr": "glukokortikoidní receptor",
    "mr": "mineralokortikoidní receptor",
    "era": "estrogenový receptor α",
    "erb": "estrogenový receptor β"
}
target_labels_cz = [czech_names.get(name.lower(), name) for name in target_names]

for i in range(n_targets):
    color = cmap(i % 10)
    plt.plot(
        n_estimators_list,
        history[i],
        marker='.',
        linestyle='-',
        linewidth=1.5,
        label=f'{target_labels_cz[i]}',
        color=color
    )

plt.title("Konvergence XGBoost: ROC-AUC vs. počet stromů", fontsize=16)
plt.xlabel("Počet stromů", fontsize=14)
plt.ylabel("ROC-AUC skóre", fontsize=14)

plt.gca().xaxis.set_major_locator(MultipleLocator(50))

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.savefig("images/xgb_convergence.svg", format="svg", bbox_inches="tight")
plt.show()